In [ ]:
import torch, shutil

# Confirm a CUDA-capable GPU is attached before proceeding.
# JPoSE inference requires a GPU; the runtime will raise AssertionError otherwise.

assert torch.cuda.is_available(), "Without GPU — activate Runtime > Change runtime type > GPU"

gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}  ({gpu.total_memory/1e9:.1f} GB VRAM)")
print(f"CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")

In [ ]:
from google.colab import drive

# Mount Google Drive so checkpoints, data, and submission files persist across sessions.

drive.mount("/content/gdrive")

GDRIVE_ROOT = "/content/gdrive/MyDrive/EK100_MIR"
print(f"Drive mounted: {GDRIVE_ROOT}")

In [ ]:
%%bash

# Install Python dependencies and download the spaCy English model.
# spaCy is used by JPoSE for POS-tag parsing of the narration text.

pip install -q pandas numpy tqdm scipy scikit-learn spacy
python -m spacy download en_core_web_sm -q 2>/dev/null || true

echo "Dependencies OK"

In [ ]:
%%bash

# Clone the Joint-Part-of-Speech-Embeddings repo if not already present.
# --depth 1 keeps the clone shallow to save time and disk space.

cd /content

if [ ! -d "Joint-Part-of-Speech-Embeddings" ]; then
    git clone -q --depth 1 https://github.com/mwray/Joint-Part-of-Speech-Embeddings.git
    echo "Cloned: Joint-Part-of-Speech-Embeddings"
else
    echo "Alredy exist: Joint-Part-of-Speech-Embeddings"
fi

In [ ]:
import subprocess, re, pickle
from pathlib import Path
import numpy as np

# ROOT is the Colab container base. JPOSE_DIR points to the cloned repo.
# GDRIVE_DATA is the data folder on Drive populated by jpose_base_data.ipynb.

ROOT        = "/content"
JPOSE_DIR   = Path(f"{ROOT}/Joint-Part-of-Speech-Embeddings")
GDRIVE_DATA = Path(f"{GDRIVE_ROOT}/data")

In [ ]:
# Internal data directories expected by the JPoSE repo layout.
# These are created when JPoSE_data.zip is extracted into JPOSE_DIR.

MODELS_DIR     = JPOSE_DIR / "data" / "models"
VID_FEAT_DIR   = JPOSE_DIR / "data" / "video_features"
TXT_FEAT_DIR   = JPOSE_DIR / "data" / "text_features"
DATAFRAMES_DIR = JPOSE_DIR / "data" / "dataframes"
RELATIONAL_DIR = JPOSE_DIR / "data" / "relational"
RELEVANCY_DIR  = JPOSE_DIR / "data" / "relevancy"

In [ ]:
# Output directories on Drive for raw submission pickles and final zip archives.
# Created here if they do not already exist.

SUBMISSIONS_DIR = Path(f"{GDRIVE_ROOT}/submissions")
ZIPS_DIR        = Path(f"{GDRIVE_ROOT}/submission_zips")
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Print a summary of all configured paths for a quick visual sanity check.

print("Configured paths OK")
print(f"  JPoSE repo : {JPOSE_DIR}")
print(f"  Drive data : {GDRIVE_DATA}")
print(f"  Submissions: {SUBMISSIONS_DIR}")
print(f"  ZIPs       : {ZIPS_DIR}")

In [ ]:
# Expected path of the JPoSE data archive on Drive.
# This zip must exist (downloaded by jpose_base_data.ipynb) before extraction can proceed.

JPOSE_ZIP = GDRIVE_DATA / "JPoSE_data.zip"

In [ ]:
# Returns True only when the model checkpoint and both feature directories
# are present and non-empty, meaning extraction has already been done.

def data_ready():

    checkpoint = MODELS_DIR / "JPoSE_BEST" / "model" / "EPIC_100_retrieval_JPoSE_BEST.pth"
    return (
        checkpoint.exists()
        and VID_FEAT_DIR.exists() and any(VID_FEAT_DIR.iterdir())
        and TXT_FEAT_DIR.exists() and any(TXT_FEAT_DIR.iterdir())
    )

In [ ]:
# Extract JPoSE_data.zip into the repo data directory if extraction has not been done yet.
# If the zip is also missing, the user is directed to run jpose_base_data.ipynb first.

if data_ready():
    print("JPoSE data available")

else:
    assert JPOSE_ZIP.exists(), (
        f"Not found {JPOSE_ZIP}\n"
        "First, run EK100_MIR_submission.ipynb to download and save the data to Drive"
    )

    print(f"Extracting {JPOSE_ZIP.name} ({JPOSE_ZIP.stat().st_size/1e9:.2f} GB)...")

    subprocess.run(
        f'unzip -o -q "{JPOSE_ZIP}" -d "{JPOSE_DIR}"',
        shell=True, check=True,
    )
    print("Complete extraction")

In [ ]:
# Verify each required data subdirectory is non-empty and print a status summary.
# Any [ERR] entry means the corresponding data was not extracted correctly.

for label, path in [
    ("models",         MODELS_DIR),
    ("video_features", VID_FEAT_DIR),
    ("text_features",  TXT_FEAT_DIR),
    ("dataframes",     DATAFRAMES_DIR),
    ("relational",     RELATIONAL_DIR),
    ("relevancy",      RELEVANCY_DIR),
]:
    files = [f for f in path.rglob("*") if f.is_file()] if path.exists() else []
    status = "OK" if files else "ERR"
    size_mb = sum(f.stat().st_size for f in files) / 1e6
    print(f"  [{status}]  {label}: {len(files)} archivo(s) ({size_mb:.0f} MB)")

checkpoint = MODELS_DIR / "JPoSE_BEST" / "model" / "EPIC_100_retrieval_JPoSE_BEST.pth"
print(f"\n  [{'OK' if checkpoint.exists() else 'ERR'}]  checkpoint: {checkpoint.name}")

In [ ]:
# Patch all torch.load() calls in the JPoSE source to pass weights_only=False.
# Required because PyTorch >= 2.0 changed the default to weights_only=True,
# which breaks JPoSE's checkpoint loading.

for fpath in JPOSE_DIR.joinpath("src").rglob("*.py"):

    txt = fpath.read_text()
    new = re.sub(
        r"torch\.load\(([^,)]+)\)",
        r"torch.load(\1, weights_only=False)",
        txt,
    )
    if new != txt:
        fpath.write_text(new)
        print(f"  Patched: {fpath.name}")

print("Patches OK")

In [ ]:
# Load the JPoSE_BEST checkpoint and print its layer names and shapes.
# This confirms the checkpoint format is compatible before running inference.

MODEL_NAME = "JPoSE_BEST"
COMB_FUNC  = "cat"

CHECKPOINT = MODELS_DIR / MODEL_NAME / "model" / f"EPIC_100_retrieval_{MODEL_NAME}.pth"
assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"

state = torch.load(str(CHECKPOINT), weights_only=False, map_location="cpu")
print(f"Checkpoint: {CHECKPOINT.name}")
print(f"Capas ({len(state)} tensores):")
for name, tensor in state.items():
    print(f"  {name:50s}  {str(tuple(tensor.shape)):20s}  {tensor.dtype}")

In [ ]:
# Run JPoSE inference on the EK-100 test split and write submission.pkl to Drive.
# PYTHONPATH is set so the script can import modules from the JPoSE src/ directory.
# Timeout is 10 minutes; increase if inference stalls on a slow runtime.

jpose_out = SUBMISSIONS_DIR / f"{MODEL_NAME}_test_latest.pkl"

print(f"Running JPoSE inference ({MODEL_NAME}, comb-func = {COMB_FUNC})...")
r = subprocess.run(
    f'cd "{JPOSE_DIR}" && PYTHONPATH="{JPOSE_DIR}/src" '
    f'python src/train/test_jpose_triplet.py "{CHECKPOINT}" '
    f'--comb-func {COMB_FUNC} --challenge-submission "{jpose_out}" 2>&1',
    shell=True, capture_output=True, text=True, timeout=600,
)

print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-600:])
    raise RuntimeError("JPoSE inference failed")

In [ ]:
# Load and validate the inference output.
# The expected shape is (9668 videos × 3842 captions); a mismatch indicates inference failed.

assert jpose_out.exists(), f"Output not generated: {jpose_out}"
with open(jpose_out, "rb") as f:
    sub = pickle.load(f)

sim_mat = np.array(sub["sim_mat"], dtype=np.float32)
vis_ids = list(sub["vis_ids"])
txt_ids = list(sub["txt_ids"])

assert sim_mat.shape == (9668, 3842), f"Shape mismatch: {sim_mat.shape}"
print(f"\nsim_mat : {sim_mat.shape}  dtype = {sim_mat.dtype}")
print(f"vis_ids : {len(vis_ids)}  |  txt_ids: {len(txt_ids)}")
print(f"Drive   : {jpose_out.name}  ({jpose_out.stat().st_size/1e6:.1f} MB)")

In [ ]:
# SLS (Supervision Level Score) values required by the Codabench grader.
# Adjust to reflect the actual supervision level of the submitted model (scale 0–5).

SLS_PT = 2
SLS_TL = 3
SLS_TD = 3
SUBMISSION_NAME = f"{MODEL_NAME}_submission"

In [ ]:
# Serialize the submission payload as a protocol-2 pickle and apply the numpy
# compatibility patch so the Codabench grader (older numpy) can deserialise it.
# numpy >= 2.0 emits 'numpy._core.multiarray' which the grader cannot unpickle.

def make_compat_pickle(sim_mat, vis_ids, txt_ids, sls_pt, sls_tl, sls_td):

    payload = {
        "version":   "0.1",
        "challenge": "multi_instance_retrieval",
        "sls_pt":    sls_pt,
        "sls_tl":    sls_tl,
        "sls_td":    sls_td,
        "sim_mat":   np.array(sim_mat, dtype=np.float32),
        "vis_ids":   [str(v) for v in vis_ids],
        "txt_ids":   [str(t) for t in txt_ids],
    }
    raw = pickle.dumps(payload, protocol=2)
    raw = raw.replace(b"numpy._core.multiarray", b"numpy.core.multiarray")
    return raw

tmp_pkl = Path("/tmp/test.pkl")
pkl_bytes = make_compat_pickle(sim_mat, vis_ids, txt_ids, SLS_PT, SLS_TL, SLS_TD)
tmp_pkl.write_bytes(pkl_bytes)

In [ ]:
# Round-trip check: reload test.pkl and confirm the sim_mat shape is preserved.
# Then package it into the final submission zip for Codabench upload.

check = pickle.loads(pkl_bytes)
assert np.array(check["sim_mat"]).shape == (9668, 3842)
print(f"test.pkl OK: {len(pkl_bytes)/1e6:.1f} MB  |  sim_mat = {np.array(check['sim_mat']).shape}")

zip_path = ZIPS_DIR / f"{SUBMISSION_NAME}.zip"
subprocess.run(
    f'cd /tmp && zip -j "{zip_path}" test.pkl',
    shell=True, check=True, capture_output=True,
)

print(f"ZIP generated : {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print(f"Drive        : {zip_path}")
print(f"\nReady to upload to: https://www.codabench.org/competitions/12008")